In [1]:
import os
import glob
from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

load_dotenv(override = True)

True

In [3]:
folders = glob.glob("knowledge-base/*")

documents = []

for folder in folders:
    doc_type = os.path.basename(folder)

    loader = DirectoryLoader(
        folder,
        glob = "**/*.md",
        loader_cls = TextLoader,
        loader_kwargs = {"encoding": "utf-8"}
    )

    folder_docs = loader.load()

    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 306 documents


In [4]:
import re

# Add legal identifiers and titles BEFORE splitting
for doc in documents:
    source = doc.metadata.get("source", "")
    filename = os.path.basename(source)

    identifier = ""
    title = ""

    article_match = re.match(r"article_(\d+)", filename)
    recital_match = re.match(r"recital_(\d+)", filename)
    annex_match = re.match(r"annex_([IVXLCDM]+)", filename)

    if article_match:
        number = int(article_match.group(1))
        identifier = f"Article {number}"

        # Extract the title appearing after "Article N"
        title_match = re.search(
            rf"## Official text\s*\n+\s*Article\s+{number}\s*\n+\s*([^\n]+)",
            doc.page_content
        )

        if title_match:
            title = title_match.group(1).strip()

    elif recital_match:
        identifier = f"Recital {int(recital_match.group(1))}"

    elif annex_match:
        identifier = f"Annex {annex_match.group(1)}"

    doc.metadata["identifier"] = identifier
    doc.metadata["title"] = title


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = text_splitter.split_documents(documents)


# Add legal identity to EVERY chunk
for chunk in chunks:
    identifier = chunk.metadata.get("identifier", "")
    title = chunk.metadata.get("title", "")

    if title:
        header = f"{identifier} — {title}"
    else:
        header = identifier

    chunk.page_content = (
        f"{header}\n\n"
        f"{chunk.page_content}"
    )


print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 1060 chunks
First chunk:

page_content='Annex I

# Annex I

## Source

Regulation (EU) 2024/1689 - Artificial Intelligence Act

## Official text

ANNEX I	 
List of Union harmonisation legislation
Section A.' metadata={'source': 'knowledge-base\\annexes\\annex_I.md', 'doc_type': 'annexes', 'identifier': 'Annex I', 'title': ''}


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)

if os.path.exists(db_name):
    Chroma(
        persist_directory = db_name,
        embedding_function = embeddings
    ).delete_collection()

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    persist_directory = db_name
)

print(
    f"Vectorstore created with "
    f"{vectorstore._collection.count()} documents"
)

Vectorstore created with 1060 documents


In [6]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(
    limit = 1,
    include = ["embeddings"]
)["embeddings"][0]

dimensions = len(sample_embedding)

print(
    f"There are {count:,} vectors with "
    f"{dimensions:,} dimensions in the vector store"
)

There are 1,060 vectors with 384 dimensions in the vector store


In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

In [8]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}
)

llm = ChatOpenAI(
    temperature = 0,
    model_name = MODEL
)

In [9]:
SYSTEM_PROMPT_TEMPLATE = """
You are an assistant answering questions about Regulation (EU) 2024/1689
(the EU Artificial Intelligence Act).

Answer the user's question using only the provided context.

Rules:
1. Base your answer only on the provided EU AI Act context.
2. Do not invent or assume legal provisions that are not supported by the context.
3. At the beginning of the answer, explicitly identify the main relevant legal
   provision using its exact identifier, for example:
   "According to Article 113..."
   "According to Recital 47..."
   "According to Annex III..."
4. If several provisions are relevant, identify the most important one first
   and mention the others where appropriate.
5. Clearly distinguish the general rule from exceptions, conditions, and
   special cases.
6. If the provided context is insufficient to answer the question, say that
   the answer cannot be determined from the provided EU AI Act materials.
7. Do not present the answer as legal advice.

Context:
{context}
"""

In [10]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)

    context = "\n\n".join(
        f"Source: {doc.metadata['source']}\n{doc.page_content}"
        for doc in docs
    )

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        context = context
    )

    response = llm.invoke([
        SystemMessage(content = system_prompt),
        HumanMessage(content = question)
    ])

    return response.content

In [11]:
answer_question(
    "When does the EU AI Act apply?",
    []
)

'According to Article 113, the EU AI Act shall apply from 2 August 2026. However, there are specific provisions that apply earlier: Chapters I and II from 2 February 2025; Chapter III Section 4, Chapter V, Chapter VII, Chapter XII, and Article 78 from 2 August 2025 (with the exception of Article 101); and Article 6(1) and the corresponding obligations from 2 August 2027.'

In [12]:
docs = retriever.invoke(
    "When does the EU AI Act apply?"
)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.metadata)
    print(doc.page_content[:800])


--- Result 1 ---
{'identifier': 'Article 113', 'title': 'Entry into force and application', 'source': 'knowledge-base\\articles\\article_113.md', 'doc_type': 'articles'}
Article 113 — Entry into force and application

# Article 113

## Source

Regulation (EU) 2024/1689 - Artificial Intelligence Act

## Official text

Article 113 
Entry into force and application
This Regulation shall enter into force on the twentieth day following that of 
its publication in the Official Journal of the European Union.
It shall apply from 2 August 2026. However:
(a)	 Chapters I and II shall apply from 2 February 2025;
(b)	 Chapter III Section 4, Chapter V, Chapter VII and Chapter XII and 
Article 78 shall apply from 2 August 2025, with the exception of 
Article 101;
(c)	 Article 6(1) and the corresponding obligations in this Regulation shall 
apply from 2 August 2027.

This Regulation shall be binding in its entirety and directly applicable in all 
Member States. Done at Br

--- Result 2 ---
{'doc_type

In [13]:
docs = vectorstore.similarity_search(
    "Which AI practices are prohibited under the EU AI Act?",
    k = 10
)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.metadata)
    print(doc.page_content[:700])


--- Result 1 ---
{'source': 'knowledge-base\\articles\\article_005.md', 'doc_type': 'articles', 'identifier': 'Article 5', 'title': 'Prohibited AI practices'}
Article 5 — Prohibited AI practices

# Article 5

## Source

Regulation (EU) 2024/1689 - Artificial Intelligence Act

## Official text

--- Result 2 ---
{'identifier': 'Article 108', 'source': 'knowledge-base\\articles\\article_108.md', 'title': 'Amendments to Regulation (EU) 2018/1139', 'doc_type': 'articles'}
Article 108 — Amendments to Regulation (EU) 2018/1139

Article 108 
Amendments to Regulation (EU) 2018/1139
Regulation (EU) 2018/1139 is amended as follows:
(1)	
in Article 17, the following paragraph is added:
‘3. Without prejudice to paragraph 2, when adopting implementing acts 
pursuant to paragraph 1 concerning Artificial Intelligence systems which 
are safety components within the meaning of Regulation (EU) 2024/1689 
of the European Parliament and of the Council (*), the requirements set 
out in Chapter III, Section

In [14]:
gr.ChatInterface(
    answer_question,
    type = "messages"
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [15]:
import importlib
import implementation.answer

importlib.reload(implementation.answer)

<module 'implementation.answer' from 'c:\\Users\\gokif\\projects\\eu_ai_act_rag\\week5\\implementation\\answer.py'>

In [16]:
import evaluation.eval

importlib.reload(evaluation.eval)

from evaluation.eval import evaluate_retrieval, evaluate_answer

In [17]:
from implementation.answer import embeddings

print(len(embeddings.embed_query("test")))


384


In [18]:
from evaluation import test

In [19]:
tests = test.load_tests()

In [20]:
len(tests)

20

In [21]:
example = tests[0]
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)

What is the purpose of the EU AI Act?
direct_fact
According to Article 1, the purpose of the EU AI Act is to improve the functioning of the internal market and promote the uptake of human-centric and trustworthy artificial intelligence while ensuring a high level of protection of health, safety and fundamental rights and supporting innovation.
['Article 1', 'internal market', 'human-centric', 'trustworthy', 'fundamental rights', 'innovation']


In [22]:
from collections import Counter
count = Counter([t.category for t in tests])
count

Counter({'obligation': 9,
         'multi_part': 4,
         'classification': 2,
         'direct_fact': 1,
         'scope': 1,
         'definition': 1,
         'transparency': 1,
         'temporal': 1})

In [23]:
from evaluation.eval import evaluate_retrieval, evaluate_answer

In [24]:
evaluate_retrieval(example)

RetrievalEval(mrr=0.08333333333333333, ndcg=0.1781035935540111, keywords_found=3, total_keywords=6, keyword_coverage=50.0)

In [25]:
eval, answer, chunks = evaluate_answer(example)

In [26]:
eval

AnswerEval(feedback='The generated answer accurately reflects the purpose of the EU AI Act as stated in the reference, including promoting trustworthy AI, protecting rights and safety, and facilitating free movement of AI products. It expands slightly on the original by mentioning additional aspects such as addressing risks in specific areas and preventing restrictions, which, while informative, are not explicitly detailed in the reference. The core purpose aligns well with the reference answer.', accuracy=5.0, completeness=4.0, relevance=5.0, legal_source_accuracy=5.0)

In [27]:
print(eval.feedback)
print(eval.accuracy)
print(eval.completeness)
print(eval.relevance)

The generated answer accurately reflects the purpose of the EU AI Act as stated in the reference, including promoting trustworthy AI, protecting rights and safety, and facilitating free movement of AI products. It expands slightly on the original by mentioning additional aspects such as addressing risks in specific areas and preventing restrictions, which, while informative, are not explicitly detailed in the reference. The core purpose aligns well with the reference answer.
5.0
4.0
5.0
